# W14C2 Lab: Wiring Workflows Instead of Agents

Run every cell from the top. **Everything already works.**

Anthropic's distinction: a WORKFLOW follows code paths you wrote, an
AGENT decides its own. Workflows are cheaper, faster and easier to debug,
so the engineering question is how far you can get without an agent.

Today you will:

1. Route each request to the right handler, and measure the saving.
2. Chain two steps so the second sees the first one's output.
3. Find the case where a workflow beats an agent, and the case where it does not.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup. Handlers are deterministic, so cost and behaviour are exactly repeatable.
import pandas as pd
import matplotlib.pyplot as plt

REQUESTS = [
    "What are your opening hours?",
    "Where is my order 4471?",
    "How do I reset my password?",
    "I want a refund for order 3320.",
    "Do you ship to Canada?",
    "My order 8890 arrived broken.",
    "What is your return policy?",
    "Reset my password please.",
]

# Pretend costs, in "model call units". Big handlers cost more.
COSTS = {"faq": 1, "order": 2, "account": 1, "escalate": 5}

def faq(text):      return "See our FAQ page.", COSTS["faq"]
def order(text):    return "Your order is in transit.", COSTS["order"]
def account(text):  return "Use the reset link on the login page.", COSTS["account"]
def escalate(text): return "Passing you to a human agent.", COSTS["escalate"]

print(f"{len(REQUESTS)} incoming requests")

## Part 1. Routing: send each request where it belongs

The simplest workflow. One cheap classifier decides which handler runs, and
most requests never touch the expensive path.

In [ ]:
# GIVEN. A router, and what it costs against sending everything to escalate.
def route(text):
    low = text.lower()
    if "order" in low:                      return order
    if "password" in low or "login" in low: return account
    if "?" in text:                         return faq
    return escalate

rows, routed_cost = [], 0
for r in REQUESTS:
    handler = route(r)
    reply, cost = handler(r)
    routed_cost += cost
    rows.append({"request": r[:36], "handler": handler.__name__, "cost": cost})

everything_escalated = COSTS["escalate"] * len(REQUESTS)
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"routed cost              : {routed_cost}")
print(f"if everything escalated  : {everything_escalated}")
print(f"saving                   : {100 * (1 - routed_cost / everything_escalated):.0f}%")

In [ ]:
# ================== YOUR TURN 1 ==================
# The router sends 'I want a refund for order 3320.' to the order
# handler, which answers 'Your order is in transit.' That is wrong: a
# refund needs a human.
#
# Fix the router so refunds escalate.
#
# Expected: the refund request routes to escalate, total cost goes up slightly,
#           and the answer becomes correct. Routing is cheap to change and easy to
#           read, which is the entire argument for doing this in code rather than
#           letting a model decide.
# ===============================================
def route_v2(text):
    low = text.lower()
    # <-- add your refund rule here, BEFORE the order rule
    if "order" in low:                      return order
    if "password" in low or "login" in low: return account
    if "?" in text:                         return faq
    return escalate

total = 0
for r in REQUESTS:
    h = route_v2(r)
    total += COSTS[h.__name__]
    flag = "  <-- refund" if "refund" in r.lower() else ""
    print(f"   {h.__name__:<10} {r[:40]}{flag}")
print(f"\ntotal cost: {total}")

## Part 2. Chaining: the second step reads the first

Prompt chaining splits one hard job into two easy ones. Each step is
simpler, and you can inspect what happened in between.

In [ ]:
# GIVEN. Extract, then act on what was extracted.
import re

def extract_order_number(text):
    match = re.search(r"\b(\d{4})\b", text)
    return match.group(1) if match else None

def lookup_status(number):
    statuses = {"4471": "in transit", "3320": "delivered", "8890": "delivered"}
    return statuses.get(number, "not found")

def chained(text):
    number = extract_order_number(text)          # step 1
    if number is None:
        return "I could not find an order number in your message."
    status = lookup_status(number)               # step 2, uses step 1's output
    return f"Order {number} is {status}."

for r in ["Where is my order 4471?", "My order 8890 arrived broken.", "Where is my stuff?"]:
    print(f"   {r:<34} -> {chained(r)}")

In [ ]:
# ================== YOUR TURN 2 ==================
# Chaining's advantage is that you can see the middle. Print what step 1
# extracted before step 2 runs, then find an input where step 1 is the
# part that fails.
#
# Expected: an order number in a different format, such as '#44-71' or 'order
#           number four four seven one', breaks the extractor while step 2 stays
#           perfectly healthy. In a single-prompt design you would only see a
#           wrong final answer and have to guess which half went wrong.
# ===============================================
TEST = "Where is my order 4471?"          # <-- try '#44-71' or a spelled-out number

number = extract_order_number(TEST)
print(f"step 1 extracted: {number!r}")
if number is None:
    print("step 2 never ran: the failure is in extraction, not lookup")
else:
    print(f"step 2 returned : {lookup_status(number)}")

## Part 3. When is an agent actually worth it?

A workflow is a fixed path. An agent chooses its own, which costs more and
fails in more ways. It earns that when you genuinely cannot enumerate the
paths in advance.

In [ ]:
# GIVEN. Same eight requests, two designs, counted.
workflow_calls = sum(COSTS[route(r).__name__] for r in REQUESTS)
agent_calls = len(REQUESTS) * 4          # decide, act, observe, answer

print(f"workflow: {workflow_calls} calls")
print(f"agent   : {agent_calls} calls   ({agent_calls / workflow_calls:.1f}x more)")

plt.figure(figsize=(4.6, 3))
plt.bar(["workflow", "agent"], [workflow_calls, agent_calls], color=["#7C2529", "#999"])
plt.ylabel("model calls"); plt.title("Cost of the same eight requests")
plt.show()
print("For requests that fall into known categories, the workflow wins outright.")
print("The agent earns its cost only when the categories cannot be listed up front.")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   Add, before the order rule:
#       if "refund" in low: return escalate
#   Order matters: the first matching rule wins, so a broader rule placed
#   earlier will swallow the cases you meant to catch later. That single fact
#   causes most routing bugs.
#
# YOUR TURN 2
#   The regex looks for exactly four digits, so '#44-71' or a spelled-out number
#   returns None and step 2 never runs. That is the point of chaining: the
#   failure is localised and visible, instead of surfacing as a wrong answer
#   from one big prompt.
#
# PART 3, nothing to edit
#   The workflow is several times cheaper here because the eight requests fall
#   into four known buckets. Start with the workflow; reach for an agent when
#   you genuinely cannot enumerate the paths, and expect to pay for it.